In [1]:
import pandas as pd

df = pd.read_csv("combined/all_years.csv", low_memory=False)
print(df.shape)

(417467, 160)


## Fix 2026 errors

In [2]:
# wrong name (only in 2026) -> correct name used by the other years
MERGE = {
    "LanguagesHaveWorkedWith":    "LanguageHaveWorkedWith",
    "LanguagesWantToWorkWith":    "LanguageWantToWorkWith",
    "DatabasesHaveWorkedWith":    "DatabaseHaveWorkedWith",
    "DatabasesWantToWorkWith":    "DatabaseWantToWorkWith",
    "Education":                  "EdLevel",
    "CloudPlatHaveWorkedWith":    "PlatformHaveWorkedWith",
    "CloudPlatWantToWorkWith":    "PlatformWantToWorkWith",
    "WebFrameworkHaveWorkedWith": "WebframeHaveWorkedWith",
    "WebFrameworkWantToWorkWith": "WebframeWantToWorkWith",
}

for wrong, right in MERGE.items():
    if wrong not in df.columns:
        print(f"skip: '{wrong}' not in file")
        continue
    if right not in df.columns:
        df = df.rename(columns={wrong: right})
        print(f"renamed {wrong} -> {right}")
        continue
    both = df[wrong].notna() & df[right].notna()
    if both.any():
        print(f"WARNING {wrong}/{right}: {both.sum()} rows have values in both, keeping '{right}'")
    df[right] = df[right].fillna(df[wrong])      # fill blanks from the misspelled column
    df = df.drop(columns=wrong)                  # then remove the misspelled column
    print(f"merged {wrong} -> {right}")


#drop the columns kept by mistake in 2026
DROP = ["TechEndorseIntro", "SOTagsWant Entry", "CommPlatformHaveEntr", "CommPlatformWantEntr"]
df = df.drop(columns=[c for c in DROP if c in df.columns])

#rename a column from 2026
if "LearnCodeHow" in df.columns:
    df = df.rename(columns={"LearnCodeHow": "LearnCode__LearnCodeHow"})


merged LanguagesHaveWorkedWith -> LanguageHaveWorkedWith
merged LanguagesWantToWorkWith -> LanguageWantToWorkWith
merged DatabasesHaveWorkedWith -> DatabaseHaveWorkedWith
merged DatabasesWantToWorkWith -> DatabaseWantToWorkWith
merged Education -> EdLevel
merged CloudPlatHaveWorkedWith -> PlatformHaveWorkedWith
merged CloudPlatWantToWorkWith -> PlatformWantToWorkWith
merged WebFrameworkHaveWorkedWith -> WebframeHaveWorkedWith
merged WebFrameworkWantToWorkWith -> WebframeWantToWorkWith


## Fill in Language count for all years

In [3]:
# number of items in the ";" list; blank answer stays blank (not 0)
df["LanguageHaveWorkedWith_count"] = (
    df["LanguageHaveWorkedWith"].str.split(";").str.len().astype("Int64")
)

## THIS CELL NEEDS TO BE RUN AT THE VERY END, add code in cells above

this notebook reads all_years.csv and applies all these changes and saves to a new file all_years_fixed.csv 

In [4]:
# checks that must pass before saving
assert df["uid"].duplicated().sum() == 0, "duplicate uids"
assert df["survey_year"].notna().all(), "blank survey_year"

df.to_csv("combined/all_years_fixed.csv", index=False)
print("saved", df.shape)

saved (417467, 147)
